# Sprint: APIs REST con Python

En este notebook practicamos el uso de APIs REST usando la librería `requests` de Python.  
Cubrimos tres niveles de dificultad:
- **Nivel 1**: API de laboratorio – JSONPlaceholder
- **Nivel 2**: API pública real – PokeAPI
- **Nivel 3**: API Open Data Barcelona (CKAN)

---

In [4]:
# Importamos las librerías que usaremos a lo largo de todo el notebook.
# 'requests' nos permite hacer peticiones HTTP (GET, POST, PATCH, DELETE...).
# 'pandas' nos sirve para convertir los datos JSON en DataFrames estructurados.
import requests
import pandas as pd

print("Librerías importadas correctamente")

Librerías importadas correctamente


---
## NIVEL 1 – Exploración básica con JSONPlaceholder

JSONPlaceholder es una API pública de laboratorio que nos permite practicar todos los métodos HTTP (GET, POST, PATCH, DELETE) sin necesidad de autenticación ni de modificar datos reales.

### Ejercicio 1 · Consultas GET a los tres recursos principales

In [5]:
# Definimos la URL base de JSONPlaceholder para no repetirla en cada petición.
url_base = "https://jsonplaceholder.typicode.com"

# Hacemos una petición GET a cada uno de los tres endpoints.
# GET devuelve los datos sin modificar nada en el servidor.
respuesta_publicaciones = requests.get(f"{url_base}/posts")
respuesta_usuarios       = requests.get(f"{url_base}/users")
respuesta_tareas         = requests.get(f"{url_base}/todos")

# Convertimos la respuesta JSON a lista de Python para poder contar elementos.
publicaciones = respuesta_publicaciones.json()
usuarios      = respuesta_usuarios.json()
tareas        = respuesta_tareas.json()

# Mostramos la cantidad total de cada recurso y el código de estado HTTP.
# 200 OK significa que la petición fue exitosa.
print("=== RESULTADOS DE LAS PETICIONES GET ===")
print(f"Publicaciones (/posts) → {len(publicaciones)} registros | Estado: {respuesta_publicaciones.status_code}")
print(f"Usuarios (/users)      → {len(usuarios)} registros      | Estado: {respuesta_usuarios.status_code}")
print(f"Tareas (/todos)        → {len(tareas)} registros   | Estado: {respuesta_tareas.status_code}")

# Mostramos un ejemplo de publicación para ver la estructura del JSON.
print("\nEjemplo de publicación (primer elemento):")
print(publicaciones[0])

=== RESULTADOS DE LAS PETICIONES GET ===
Publicaciones (/posts) → 100 registros | Estado: 200
Usuarios (/users)      → 10 registros      | Estado: 200
Tareas (/todos)        → 200 registros   | Estado: 200

Ejemplo de publicación (primer elemento):
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### Ejercicio 2 · Petición a un recurso inexistente → Error 404

In [7]:
# Pedimos una publicación con un ID que no existe (99999).
# JSONPlaceholder devolverá un 404 Not Found, que es el código estándar cuando el servidor no encuentra el recurso solicitado.
respuesta_no_encontrada = requests.get(f"{url_base}/posts/99999")

print("=== PETICIÓN A RECURSO INEXISTENTE ===")
print(f"URL consultada: {respuesta_no_encontrada.url}")
print(f"Código de estado: {respuesta_no_encontrada.status_code}")
print(f"Respuesta del servidor: {respuesta_no_encontrada.text}")

=== PETICIÓN A RECURSO INEXISTENTE ===
URL consultada: https://jsonplaceholder.typicode.com/posts/99999
Código de estado: 404
Respuesta del servidor: {}


### Ejercicio 3 · Petición POST – Crear una nueva publicación

In [8]:
# Con POST enviamos datos al servidor para CREAR un nuevo recurso.
# En JSONPlaceholder no se guarda de verdad, pero simula el comportamiento real.
# Definimos el cuerpo de la nueva publicación como un diccionario Python.
nueva_publicacion = {
    "title":  "Mi primera publicación de prueba",
    "body":   "Este es el contenido del mensaje creado con una petición POST.",
    "userId": 1
}

# Enviamos la petición POST con el parámetro 'json=' para que requests serialice el diccionario automáticamente y añada el header Content-Type.
respuesta_post = requests.post(f"{url_base}/posts", json=nueva_publicacion)

print("=== RESULTADO DE LA PETICIÓN POST ===")
print(f"Código de estado: {respuesta_post.status_code}")  # 201 Created = éxito
print("Respuesta JSON del servidor:")
print(respuesta_post.json())

=== RESULTADO DE LA PETICIÓN POST ===
Código de estado: 201
Respuesta JSON del servidor:
{'title': 'Mi primera publicación de prueba', 'body': 'Este es el contenido del mensaje creado con una petición POST.', 'userId': 1, 'id': 101}


### Ejercicio 4 · Petición PATCH – Modificar parcialmente una publicación

In [9]:
# PATCH actualiza PARCIALMENTE un recurso existente: solo enviamos el campo que queremos cambiar, sin tocar el resto.
# (A diferencia de PUT, que reemplaza el recurso completo.)
cambio_parcial = {
    "title": "Título actualizado con PATCH"
}

# Modificamos la publicación con id=1.
respuesta_patch = requests.patch(f"{url_base}/posts/1", json=cambio_parcial)

print("=== RESULTADO DE LA PETICIÓN PATCH ===")
print(f"Código de estado: {respuesta_patch.status_code}")  # 200 OK
print("Respuesta JSON del servidor:")
print(respuesta_patch.json())

=== RESULTADO DE LA PETICIÓN PATCH ===
Código de estado: 200
Respuesta JSON del servidor:
{'userId': 1, 'id': 1, 'title': 'Título actualizado con PATCH', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### Ejercicio 5 · Petición DELETE – Eliminar una publicación

In [10]:
# DELETE elimina el recurso indicado del servidor.
# En una API real esto borraría el dato; JSONPlaceholder lo simula.
# Eliminamos la publicación con id=1.
respuesta_delete = requests.delete(f"{url_base}/posts/1")

print("=== RESULTADO DE LA PETICIÓN DELETE ===")
print(f"Código de estado: {respuesta_delete.status_code}")  # 200 OK
# Tras un DELETE exitoso el servidor suele devolver un body vacío o {}.
print(f"Respuesta JSON del servidor: {respuesta_delete.json()}")

=== RESULTADO DE LA PETICIÓN DELETE ===
Código de estado: 200
Respuesta JSON del servidor: {}


---
## NIVEL 2 – API pública real: PokeAPI

### Documentación de la API elegida

He elegido **[PokeAPI](https://pokeapi.co/)** porque:
- Es completamente gratuita y no requiere autenticación.
- Devuelve respuestas en formato JSON.
- Tiene múltiples endpoints bien documentados.

#### Endpoints disponibles (mínimo dos):

| Endpoint | Descripción |
|---|---|
| `GET /pokemon` | Lista de todos los Pokémon disponibles |
| `GET /pokemon/{name}` | Detalles de un Pokémon específico (stats, tipos, habilidades...) |
| `GET /type` | Lista de todos los tipos de Pokémon (fuego, agua, planta...) |
| `GET /ability` | Lista de todas las habilidades disponibles |

#### Filtros y parámetros opcionales:

| Parámetro | Descripción | Ejemplo |
|---|---|---|
| `limit` | Número máximo de resultados a devolver | `?limit=20` |
| `offset` | Desde qué posición empezar (paginación) | `?offset=0` |

**Ejemplo combinado**: `https://pokeapi.co/api/v2/pokemon?limit=50&offset=0`  
→ Devuelve los primeros 50 Pokémon.

### Ejercicio 1 · Petición GET sencilla a PokeAPI

In [11]:
# URL base de PokeAPI.
url_pokeapi = "https://pokeapi.co/api/v2"

# Hacemos una petición GET al endpoint /pokemon con parámetros de filtro.
# 'params' es un diccionario que requests convierte automáticamente en query string (?limit=20&offset=0).
parametros = {"limit": 20, "offset": 0}
respuesta_pokemon = requests.get(f"{url_pokeapi}/pokemon", params=parametros)

print("=== PETICIÓN GET A POKEAPI ===")
print(f"URL final consultada: {respuesta_pokemon.url}")
print(f"Código de estado: {respuesta_pokemon.status_code}")

# Convertimos la respuesta a diccionario Python.
datos_pokemon = respuesta_pokemon.json()

# Imprimimos algunos campos relevantes de la respuesta.
print(f"\nTotal de Pokémon disponibles en la API: {datos_pokemon['count']}")
print(f"Pokémon recibidos en esta página: {len(datos_pokemon['results'])}")
print("\nPrimeros 5 Pokémon de la lista:")
for pokemon in datos_pokemon["results"][:5]:
    print(f"  - {pokemon['name']} | URL: {pokemon['url']}")

=== PETICIÓN GET A POKEAPI ===
URL final consultada: https://pokeapi.co/api/v2/pokemon?limit=20&offset=0
Código de estado: 200

Total de Pokémon disponibles en la API: 1350
Pokémon recibidos en esta página: 20

Primeros 5 Pokémon de la lista:
  - bulbasaur | URL: https://pokeapi.co/api/v2/pokemon/1/
  - ivysaur | URL: https://pokeapi.co/api/v2/pokemon/2/
  - venusaur | URL: https://pokeapi.co/api/v2/pokemon/3/
  - charmander | URL: https://pokeapi.co/api/v2/pokemon/4/
  - charmeleon | URL: https://pokeapi.co/api/v2/pokemon/5/


### Ejercicio 2 · Consultar detalles de un Pokémon específico

In [ ]:
# Consultamos el endpoint /pokemon/{name} para obtener los detalles de Pikachu.
# Este endpoint devuelve información muy detallada: estadísticas, tipos, habilidades, peso, altura, movimientos, etc.
respuesta_pikachu = requests.get(f"{url_pokeapi}/pokemon/pikachu")

print(f"Código de estado: {respuesta_pikachu.status_code}")

detalles_pikachu = respuesta_pikachu.json()

# Mostramos solo los campos más relevantes para no saturar la salida.
print(f"\nNombre:  {detalles_pikachu['name']}")
print(f"ID:      {detalles_pikachu['id']}")
print(f"Altura:  {detalles_pikachu['height']} decímetros")
print(f"Peso:    {detalles_pikachu['weight']} hectogramos")
print(f"Tipos:   {[t['type']['name'] for t in detalles_pikachu['types']]}")
print(f"Habilidades: {[a['ability']['name'] for a in detalles_pikachu['abilities']]}")
print("Estadísticas base:")
for stat in detalles_pikachu["stats"]:
    print(f"  {stat['stat']['name']:20s}: {stat['base_stat']}")

Código de estado: 200

Nombre:  pikachu
ID:      25
Altura:  4 decímetros
Peso:    60 hectogramos
Tipos:   ['electric']
Habilidades: ['static', 'lightning-rod']
Estadísticas base:
  hp                  : 35
  attack              : 55
  defense             : 40
  special-attack      : 50
  special-defense     : 50
  speed               : 90


### Ejercicio 3 · Convertir la lista de Pokémon a un DataFrame de pandas

In [13]:
# Pedimos más Pokémon para tener un DataFrame más interesante.
# Usamos limit=151 para obtener la primera generación completa.
parametros_gen1 = {"limit": 151, "offset": 0}
respuesta_gen1 = requests.get(f"{url_pokeapi}/pokemon", params=parametros_gen1)
lista_pokemon = respuesta_gen1.json()["results"]

# La lista solo contiene 'name' y 'url'. Añadimos el ID extrayéndolo de la URL.
# La URL tiene el formato: https://pokeapi.co/api/v2/pokemon/25/
# Hacemos un pequeño procesamiento para extraer el número del ID.
for pokemon in lista_pokemon:
    pokemon["id"] = int(pokemon["url"].rstrip("/").split("/")[-1])

# Creamos el DataFrame directamente desde la lista de diccionarios y con pandas convirtimos listas de dicts a tablas estructuradas.
df_pokemon = pd.DataFrame(lista_pokemon)

# Reordenamos las columnas para que sea más legible.
df_pokemon = df_pokemon[["id", "name", "url"]]

print("=== DATAFRAME DE POKÉMON (Generación 1) ===")
print(f"Filas: {df_pokemon.shape[0]} | Columnas: {df_pokemon.shape[1]}")
print("\nPrimeras 10 filas:")
df_pokemon.head(10)

=== DATAFRAME DE POKÉMON (Generación 1) ===
Filas: 151 | Columnas: 3

Primeras 10 filas:


,id,name,url
0,1,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/
1,2,ivysaur,https://pokeapi.co/api/v2/pokemon/2/
2,3,venusaur,https://pokeapi.co/api/v2/pokemon/3/
3,4,charmander,https://pokeapi.co/api/v2/pokemon/4/
4,5,charmeleon,https://pokeapi.co/api/v2/pokemon/5/
5,6,charizard,https://pokeapi.co/api/v2/pokemon/6/
6,7,squirtle,https://pokeapi.co/api/v2/pokemon/7/
7,8,wartortle,https://pokeapi.co/api/v2/pokemon/8/
8,9,blastoise,https://pokeapi.co/api/v2/pokemon/9/
9,10,caterpie,https://pokeapi.co/api/v2/pokemon/10/


---
## NIVEL 3 – API Open Data Barcelona (CKAN)

El Ayuntamiento de Barcelona ofrece su portal de datos abiertos a través de la API **CKAN**.  
Los tres endpoints principales que usaremos son:

| Acción CKAN | Descripción |
|---|---|
| `package_search` | Busca datasets por palabra clave |
| `package_show` | Muestra todos los detalles de un dataset (incluyendo sus recursos) |
| `datastore_search` | Consulta los datos reales almacenados en un recurso |

Vamos a trabajar con el dataset de **accidentes de tráfico** gestionados por la Guardia Urbana.

### Ejercicio 1 · Buscar un dataset con `package_search`

In [21]:
url_bcn = "https://opendata-ajuntament.barcelona.cat/data/api/action"

# Usamos el término en catalán 'accidents' que devuelve 6 datasets relevantes.
# La query 'accidents transit' no devuelve resultados porque el portal
# indexa los títulos en catalán e inglés, no en español.
parametros_busqueda = {
    "q":    "accidents",
    "rows": 6
}

respuesta_busqueda = requests.get(
    f"{url_bcn}/package_search",
    params=parametros_busqueda
)

print(f"Código de estado: {respuesta_busqueda.status_code}")
datos_busqueda = respuesta_busqueda.json()

print(f"Búsqueda exitosa: {datos_busqueda['success']}")
print(f"Total de datasets encontrados: {datos_busqueda['result']['count']}")
print("\nDatasets encontrados:")
for i, dataset in enumerate(datos_busqueda["result"]["results"]):
    print(f"  [{i}] ID: {dataset['name']} | Título: {dataset['title']}")

Código de estado: 200
Búsqueda exitosa: True
Total de datasets encontrados: 6

Datasets encontrados:
  [0] ID: accidents_causa_conductor_gu_bcn | Título: Accidents by driver cause managed by the Guàrdia Urbana in the city of Barcelona
  [1] ID: accidents-causes-gu-bcn | Título: Accidents by mediate cause managed by the Guàrdia Urbana in the city of Barcelona
  [2] ID: accidents-tipus-gu-bcn | Título: Accidents  by type managed by the Guàrdia Urbana in the city of Barcelona 
  [3] ID: accidents-persones-gu-bcn | Título: People involved in accidents managed by the Police in the city of Barcelona 
  [4] ID: accidents-vehicles-gu-bcn | Título: Vehicles involved in accidents handled by the police in the city of Barcelona 
  [5] ID: accidents-gu-bcn | Título: Accidents managed by the local police in the city of Barcelona


### Ejercicio 2 · Obtener los detalles del dataset con `package_show`

In [22]:
# Seleccionamos el dataset general de accidentes [5].
nombre_dataset = datos_busqueda["result"]["results"][5]["name"]
print(f"Dataset seleccionado: {nombre_dataset}")

respuesta_detalle = requests.get(
    f"{url_bcn}/package_show",
    params={"id": nombre_dataset}
)

datos_detalle = respuesta_detalle.json()
dataset_info = datos_detalle["result"]

print(f"Título: {dataset_info['title']}")
print(f"Número de recursos disponibles: {len(dataset_info['resources'])}")
print("\nRecursos disponibles:")
for i, recurso in enumerate(dataset_info["resources"]):
    print(f"  [{i}] Formato: {recurso.get('format','?'):8s} | "
          f"Nombre: {recurso.get('name','Sin nombre')} | "
          f"ID: {recurso.get('id','')}")

Dataset seleccionado: accidents-gu-bcn
Título: Accidents managed by the local police in the city of Barcelona
Número de recursos disponibles: 22

Recursos disponibles:
  [0] Formato: CSV      | Nombre: 2025_accidents_gu_bcn.csv | ID: 066d46b1-25be-4f08-b0e0-5a233714bda2
  [1] Formato: CSV      | Nombre: 2024_accidents_gu_bcn.csv | ID: 66f5e7e3-045b-4d19-b649-2eaea622ae93
  [2] Formato: CSV      | Nombre: 2023_ACCIDENTS_GU_BCN.csv | ID: d12c8a6f-feb0-4c3e-bcf1-46b8a8b3d651
  [3] Formato: CSV      | Nombre: 2022_ACCIDENTS_GU_BCN.csv | ID: 4d41914e-3121-40ee-9f8a-23c2ffe0d560
  [4] Formato: CSV      | Nombre: 2021_ACCIDENTS_GU_BCN.csv | ID: 2b689b1f-5653-4ee8-b378-76ad5a336423
  [5] Formato: CSV      | Nombre: 2020_ACCIDENTS_GU_BCN.csv | ID: bcfc0866-7e2a-4054-9a93-fb3f371fc5eb
  [6] Formato: CSV      | Nombre: 2019_ACCIDENTS_GU_BCN.CSV | ID: 7a376881-0767-4ec2-bb41-12f4b645f1df
  [7] Formato: CSV      | Nombre: 2018_accidents_gu_bcn.csv | ID: f94d9ac3-e46e-47cd-a3d0-a9b5b9639d86
  [8] Fo

### Ejercicio 3 · Seleccionar un recurso CSV/JSON y obtener su `resource_id`

In [23]:
# Buscamos automáticamente el primer recurso que sea CSV o JSON.
# Estos formatos son los que tienen soporte para datastore_search.
id_recurso = None
nombre_recurso = None

for recurso in dataset_info["resources"]:
    formato = recurso.get("format", "").upper()
    if formato in ["CSV", "JSON"]:
        id_recurso     = recurso["id"]
        nombre_recurso = recurso.get("name", "Sin nombre")
        break  # Tomamos el primero que encontremos

if id_recurso:
    print(f"Recurso seleccionado: {nombre_recurso}")
    print(f"resource_id: {id_recurso}")
else:
    print("No se encontró ningún recurso CSV o JSON en este dataset.")
    print("Prueba a cambiar el índice del dataset en la celda anterior.")

Recurso seleccionado: 2025_accidents_gu_bcn.csv
resource_id: 066d46b1-25be-4f08-b0e0-5a233714bda2


### Ejercicio 4 · Consultar los datos con `datastore_search` (mínimo 100 registros)

In [24]:
# datastore_search nos permite consultar directamente los registros del recurso.
# Es como hacer una consulta SQL pero mediante una petición HTTP.
# El parámetro 'limit' controla cuántos registros queremos (mínimo 100).
parametros_datos = {
    "resource_id": id_recurso,
    "limit":       100,
    "offset":      0      # Empezamos desde el primer registro
}

respuesta_datos = requests.get(
    f"{url_bcn}/datastore_search",
    params=parametros_datos
)

print(f"Código de estado: {respuesta_datos.status_code}")
datos_json = respuesta_datos.json()

# La respuesta tiene la estructura: {'success': True, 'result': {'records': [...]}}
print(f"Petición exitosa: {datos_json['success']}")
registros = datos_json["result"]["records"]
print(f"Registros recibidos: {len(registros)}")

# Mostramos el primer registro como ejemplo de la estructura.
if registros:
    print("\nEjemplo del primer registro:")
    print(registros[0])

Código de estado: 200
Petición exitosa: True
Registros recibidos: 100

Ejemplo del primer registro:
{'Nom_barri': 'el Poble Sec', 'Codi_districte': '3', 'Codi_carrer': '700690', 'Coordenada_UTM_Y_ED50': '429920.025', 'Mes_any': '1', 'Descripcio_torn': 'Nit', 'NK_Any': '2025', 'Numero_lesionats_greus': None, 'Numero_lesionats_lleus': None, 'Numero_victimes': None, 'Numero_vehicles_implicats': '1', 'Nom_districte': 'Sants-Montjuïc', 'Num_postal': '187', 'Nom_mes': 'Gener', 'Nom_carrer': 'Migdia', 'Numero_morts': None, 'Latitud_WGS84': '41.36198056', 'Longitud_WGS84': '2.16099940', 'Coordenada_UTM_X_ED50': '4579485.471', 'Dia_mes': '1', 'Hora_dia': '4', 'Numero_expedient': '2025S000001', 'Codi_barri': '11', '_id': 1, 'Descripcio_dia_setmana': 'Dimecres', 'Descripcio_causa_vianant': ''}


### Ejercicio 5 · Convertir los resultados a un DataFrame de pandas

In [25]:
# Convertimos directamente la lista de registros a un DataFrame.
# Cada registro es un diccionario, y pandas los transforma en filas.
df_accidentes = pd.DataFrame(registros)

# Eliminamos la columna '_id' que añade CKAN internamente y no aporta información.
if "_id" in df_accidentes.columns:
    df_accidentes = df_accidentes.drop(columns=["_id"])

print("=== DATAFRAME DE ACCIDENTES DE TRÁFICO BCN ===")
print(f"Dimensiones: {df_accidentes.shape[0]} filas × {df_accidentes.shape[1]} columnas")
print(f"Columnas disponibles: {list(df_accidentes.columns)}")
print("\nPrimeras 5 filas:")
df_accidentes.head(5)

=== DATAFRAME DE ACCIDENTES DE TRÁFICO BCN ===
Dimensiones: 100 filas × 25 columnas
Columnas disponibles: ['Nom_barri', 'Codi_districte', 'Codi_carrer', 'Coordenada_UTM_Y_ED50', 'Mes_any', 'Descripcio_torn', 'NK_Any', 'Numero_lesionats_greus', 'Numero_lesionats_lleus', 'Numero_victimes', 'Numero_vehicles_implicats', 'Nom_districte', 'Num_postal', 'Nom_mes', 'Nom_carrer', 'Numero_morts', 'Latitud_WGS84', 'Longitud_WGS84', 'Coordenada_UTM_X_ED50', 'Dia_mes', 'Hora_dia', 'Numero_expedient', 'Codi_barri', 'Descripcio_dia_setmana', 'Descripcio_causa_vianant']

Primeras 5 filas:


,Nom_barri,Codi_districte,Codi_carrer,Coordenada_UTM_Y_ED50,Mes_any,Descripcio_torn,NK_Any,Numero_lesionats_greus,Numero_lesionats_lleus,Numero_victimes,...,Numero_morts,Latitud_WGS84,Longitud_WGS84,Coordenada_UTM_X_ED50,Dia_mes,Hora_dia,Numero_expedient,Codi_barri,Descripcio_dia_setmana,Descripcio_causa_vianant
0,el Poble Sec,3,700690,429920.025,1,Nit,2025,None,None,None,...,None,41.36198056,2.16099940,4579485.471,1,4,2025S000001,11,Dimecres,
1,el Clot,10,18505,432504.379,1,Matí,2025,None,1,1,...,None,41.41197011,2.19127893,4585011.175,1,11,2025S000002,65,Dimecres,
2,el Parc i la Llacuna del Poblenou,10,11200,432383.518,1,Matí,2025,None,1,1,...,None,41.39721760,2.19001592,4583374.358,1,12,2025S000003,66,Dimecres,
3,Sant Gervasi - Galvany,5,23403,429100.957,1,Matí,2025,None,2,2,...,None,41.39499324,2.15077769,4583158.797,2,7,2025S000004,26,Dijous,Creuar per fora pas de vianants
4,les Corts,4,700721,427208.073,1,Tarda,2025,None,None,None,...,None,41.38181302,2.12831108,4581714.187,1,20,2025S000005,19,Dimecres,


In [26]:
# Mostramos información general del DataFrame: tipos de datos y valores nulos.
# Esto nos ayuda a entender la calidad y estructura de los datos recibidos.
print("Información del DataFrame:")
df_accidentes.info()

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Nom_barri                  100 non-null    object
 1   Codi_districte             100 non-null    object
 2   Codi_carrer                100 non-null    object
 3   Coordenada_UTM_Y_ED50      100 non-null    object
 4   Mes_any                    100 non-null    object
 5   Descripcio_torn            100 non-null    object
 6   NK_Any                     100 non-null    object
 7   Numero_lesionats_greus     5 non-null      object
 8   Numero_lesionats_lleus     87 non-null     object
 9   Numero_victimes            91 non-null     object
 10  Numero_vehicles_implicats  100 non-null    object
 11  Nom_districte              100 non-null    object
 12  Num_postal                 100 non-null    object
 13  Nom_mes                    100 non-null

### Ejercicio 6 · Guardar el DataFrame en un fichero CSV

In [27]:
# Guardamos el DataFrame como CSV con codificación UTF-8 y sin el índice numérico.
# 'index=False' evita que pandas añada una columna extra con los números de fila.
# 'encoding="utf-8-sig"' asegura que los acentos y ñ se lean bien en Excel.
ruta_csv = "accidentes_trafico_barcelona.csv"
df_accidentes.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

print(f"DataFrame guardado correctamente en: '{ruta_csv}'")
print(f"Tamaño del archivo: {pd.read_csv(ruta_csv).shape}")
print("\nPrimeras 3 filas del CSV leído de nuevo para verificar:")
pd.read_csv(ruta_csv).head(3)

DataFrame guardado correctamente en: 'accidentes_trafico_barcelona.csv'
Tamaño del archivo: (100, 25)

Primeras 3 filas del CSV leído de nuevo para verificar:


,Nom_barri,Codi_districte,Codi_carrer,Coordenada_UTM_Y_ED50,Mes_any,Descripcio_torn,NK_Any,Numero_lesionats_greus,Numero_lesionats_lleus,Numero_victimes,...,Numero_morts,Latitud_WGS84,Longitud_WGS84,Coordenada_UTM_X_ED50,Dia_mes,Hora_dia,Numero_expedient,Codi_barri,Descripcio_dia_setmana,Descripcio_causa_vianant
0,el Poble Sec,3,700690,429920.025,1,Nit,2025,NaN,NaN,NaN,...,NaN,41.361981,2.160999,4579485.471,1,4,2025S000001,11,Dimecres,NaN
1,el Clot,10,18505,432504.379,1,Matí,2025,NaN,1.0,1.0,...,NaN,41.411970,2.191279,4585011.175,1,11,2025S000002,65,Dimecres,NaN
2,el Parc i la Llacuna del Poblenou,10,11200,432383.518,1,Matí,2025,NaN,1.0,1.0,...,NaN,41.397218,2.190016,4583374.358,1,12,2025S000003,66,Dimecres,NaN
